## Carga y Exploración del Dataset

En esta etapa se cargó el dataset de Kindle Reviews utilizando Pandas.

El objetivo fue explorar la información disponible y verificar que los datos se cargaran correctamente antes del preprocesamiento y entrenamiento de modelos.

El dataset contiene reseñas de usuarios, calificaciones y texto asociado a cada review.

## Vista Inicial del Dataset

Se visualizaron las primeras filas del dataset para identificar las columnas disponibles y comprender la estructura de los datos.

Entre las columnas más importantes se encuentran:
- `reviewText`
- `sentiment`
- `tokens`
- `stemmed`
- `lemmatized`

Estas columnas serán utilizadas posteriormente para el análisis de sentimientos y entrenamiento de modelos NLP.


In [ ]:
import pandas as pd
import ast

df = pd.read_csv('/content/kindle_reviews_processed.csv', on_bad_lines='skip')

df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,sentiment,clean_review,tokens,tokens_no_stopwords,stemmed,lemmatized
0,A1F6404F1VG29J,B000F83SZQ,Avidreader,"[0, 0]",I enjoy vintage books and movies so I enjoyed ...,5.0,Nice vintage story,1399248000,"05 5, 2014",1,i enjoy vintage books and movies so i enjoyed ...,"['enjoy', 'vintage', 'books', 'movies', 'enjoy...","['enjoy', 'vintage', 'books', 'movies', 'enjoy...","['enjoy', 'vintag', 'book', 'movi', 'enjoy', '...","['enjoy', 'vintage', 'book', 'movie', 'enjoyed..."
1,AN0N05A9LIJEQ,B000F83SZQ,critters,"[2, 2]",This book is a reissue of an old one; the auth...,4.0,Different...,1388966400,"01 6, 2014",1,this book is a reissue of an old one the autho...,"['book', 'reissue', 'old', 'one', 'author', 'b...","['book', 'reissue', 'old', 'one', 'author', 'b...","['book', 'reissu', 'old', 'one', 'author', 'bo...","['book', 'reissue', 'old', 'one', 'author', 'b..."
2,A795DMNCJILA6,B000F83SZQ,dot,"[2, 2]",This was a fairly interesting read. It had ol...,4.0,Oldie,1396569600,"04 4, 2014",1,this was a fairly interesting read it had old...,"['fairly', 'interesting', 'read', 'old', 'styl...","['fairly', 'interesting', 'read', 'old', 'styl...","['fairli', 'interest', 'read', 'old', 'style',...","['fairly', 'interesting', 'read', 'old', 'styl..."
3,A1FV0SX13TWVXQ,B000F83SZQ,"Elaine H. Turley ""Montana Songbird""","[1, 1]",I'd never read any of the Amy Brewster mysteri...,5.0,I really liked it.,1392768000,"02 19, 2014",1,id never read any of the amy brewster mysterie...,"['id', 'never', 'read', 'amy', 'brewster', 'my...","['id', 'never', 'read', 'amy', 'brewster', 'my...","['id', 'never', 'read', 'ami', 'brewster', 'my...","['id', 'never', 'read', 'amy', 'brewster', 'my..."
4,A3SPTOKDG7WBLN,B000F83SZQ,Father Dowling Fan,"[0, 1]","If you like period pieces - clothing, lingo, y...",4.0,Period Mystery,1395187200,"03 19, 2014",1,if you like period pieces clothing lingo you ...,"['like', 'period', 'pieces', 'clothing', 'ling...","['like', 'period', 'pieces', 'clothing', 'ling...","['like', 'period', 'piec', 'cloth', 'lingo', '...","['like', 'period', 'piece', 'clothing', 'lingo..."


## Conversión de Tokens a Texto

Las listas de palabras procesadas mediante stemming y lemmatization fueron convertidas nuevamente a texto plano.

Esto fue necesario porque los modelos de Machine Learning y las técnicas de vectorización requieren texto continuo como entrada.

In [ ]:
df = df.dropna(subset=["stemmed", "lemmatized", "sentiment"])

def convertir_lista_a_texto(x):
    try:
        return " ".join(ast.literal_eval(x))
    except:
        return ""

df["stemmed_text"] = df["stemmed"].apply(convertir_lista_a_texto)
df["lemmatized_text"] = df["lemmatized"].apply(convertir_lista_a_texto)

df[["stemmed_text", "lemmatized_text", "sentiment"]].head()

,stemmed_text,lemmatized_text,sentiment
0,enjoy vintag book movi enjoy read book plot un...,enjoy vintage book movie enjoyed reading book ...,1
1,book reissu old one author born era say nero w...,book reissue old one author born era say nero ...,1
2,fairli interest read old style terminologyi gl...,fairly interesting read old style terminologyi...,1
3,id never read ami brewster mysteri one realli ...,id never read amy brewster mystery one really ...,1
4,like period piec cloth lingo enjoy mysteri aut...,like period piece clothing lingo enjoy mystery...,1


## Definición de Variables

Se definieron las variables principales para el entrenamiento del modelo.

La variable `X` contiene el texto procesado mediante lemmatization, mientras que la variable `y` contiene las etiquetas de sentimiento.

Estas variables serán utilizadas para entrenar y evaluar los modelos de clasificación.

In [ ]:
X = df["lemmatized_text"]
y = df["sentiment"]

print(X.head())
print(y.head())


0    enjoy vintage book movie enjoyed reading book ...
1    book reissue old one author born era say nero ...
2    fairly interesting read old style terminologyi...
3    id never read amy brewster mystery one really ...
4    like period piece clothing lingo enjoy mystery...
Name: lemmatized_text, dtype: object
0    1
1    1
2    1
3    1
4    1
Name: sentiment, dtype: int64


## División de Datos

El dataset fue dividido en conjuntos de entrenamiento y prueba utilizando la función `train_test_split`.

Se utilizó:
- 80% de los datos para entrenamiento
- 20% para pruebas

Además, se aplicó `stratify=y` para mantener la misma proporción de clases positivas y negativas en ambos conjuntos.

Esta división permite evaluar el desempeño del modelo con datos que no fueron utilizados durante el entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 7503
Test: 1876


## Representación Vectorial: Bag of Words (BoW)

Se utilizó la técnica Bag of Words para transformar el texto en representaciones numéricas.

Esta técnica convierte cada palabra en una característica dentro de una matriz de frecuencias, permitiendo que los modelos de Machine Learning puedan procesar texto.

La matriz generada contiene:
- filas correspondientes a las reseñas
- columnas correspondientes al vocabulario detectado

Se identificaron más de 31 mil palabras únicas dentro del dataset.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

print(X_train_bow.shape)
print(X_test_bow.shape)

(7503, 31004)
(1876, 31004)


```markdown id="4r5x89"
## Representación Vectorial: TF-IDF

Se implementó TF-IDF como una alternativa a Bag of Words para representar el texto de forma numérica.

TF-IDF asigna mayor importancia a palabras relevantes y reduce el peso de palabras demasiado frecuentes dentro del dataset.

Esto permite obtener una representación más informativa del texto y mejorar el desempeño de los modelos de clasificación.

La matriz generada conserva:
- filas correspondientes a las reseñas
- columnas correspondientes al vocabulario detectado
```



## Resultado de TF-IDF

La técnica TF-IDF generó una matriz con dimensiones similares a Bag of Words, manteniendo más de 31 mil palabras dentro del vocabulario detectado.

Sin embargo, TF-IDF mejora la representación del texto al asignar distintos pesos a las palabras según su importancia dentro del dataset.

Esto permite reducir el impacto de palabras demasiado frecuentes y destacar términos más relevantes para la clasificación de sentimientos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(7503, 31004)
(1876, 31004)


## Entrenamiento de Logistic Regression con Bag of Words

Se entrenó un modelo de Logistic Regression utilizando la representación Bag of Words.

El objetivo fue clasificar automáticamente las reseñas como positivas o negativas a partir de las palabras presentes en cada review.

Posteriormente, el modelo realizó predicciones sobre el conjunto de prueba para evaluar su desempeño en el análisis de sentimientos.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_bow = LogisticRegression(max_iter=1000)

lr_bow.fit(X_train_bow, y_train)

pred_lr_bow = lr_bow.predict(X_test_bow)

print(pred_lr_bow[:10])

[1 1 1 1 1 1 0 1 0 1]


## Evaluación de Logistic Regression con Bag of Words

El modelo fue evaluado utilizando métricas de clasificación para medir su desempeño en el análisis de sentimientos.

Los resultados obtenidos muestran un accuracy aproximado del 82%, indicando un buen desempeño general en la clasificación de reseñas.

Además, el modelo logró identificar de mejor manera las reseñas positivas en comparación con las negativas, debido al desbalance presente en el dataset.

También se utilizó una matriz de confusión para visualizar los aciertos y errores de clasificación realizados por el modelo.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:")
print(accuracy_score(y_test, pred_lr_bow))

print("\nClassification Report:")
print(classification_report(y_test, pred_lr_bow))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_lr_bow))

Accuracy:
0.82409381663113

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.54      0.60       454
           1       0.86      0.92      0.89      1422

    accuracy                           0.82      1876
   macro avg       0.77      0.73      0.74      1876
weighted avg       0.81      0.82      0.82      1876


Confusion Matrix:
[[ 243  211]
 [ 119 1303]]


## Entrenamiento de Random Forest con Bag of Words

En esta etapa se utilizó el algoritmo Random Forest para realizar la clasificación de sentimientos utilizando las características obtenidas mediante Bag of Words.

El modelo fue entrenado con los datos de entrenamiento y posteriormente se generaron predicciones sobre el conjunto de prueba.

Finalmente, se imprimieron las primeras predicciones realizadas por el modelo para verificar su funcionamiento.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_bow = RandomForestClassifier(random_state=42)

rf_bow.fit(X_train_bow, y_train)

pred_rf_bow = rf_bow.predict(X_test_bow)

print(pred_rf_bow[:10])

[1 1 1 1 1 1 1 1 1 1]


## Evaluación del modelo Random Forest con Bag of Words

Después del entrenamiento, se evaluó el desempeño del modelo Random Forest utilizando métricas de clasificación.

El modelo obtuvo una exactitud aproximada del 79%, lo que indica un desempeño aceptable en la clasificación de sentimientos.

Además, se analizaron métricas como precisión, recall y f1-score para cada clase, así como la matriz de confusión, permitiendo observar el comportamiento del modelo en las predicciones positivas y negativas.

In [ ]:
print("Accuracy:")
print(accuracy_score(y_test, pred_rf_bow))

print("\nClassification Report:")
print(classification_report(y_test, pred_rf_bow))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_rf_bow))

Accuracy:
0.7910447761194029

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.16      0.27       454
           1       0.79      0.99      0.88      1422

    accuracy                           0.79      1876
   macro avg       0.83      0.58      0.57      1876
weighted avg       0.81      0.79      0.73      1876


Confusion Matrix:
[[  72  382]
 [  10 1412]]


## Entrenamiento de Regresión Logística con TF-IDF

En esta sección se entrenó un modelo de Regresión Logística utilizando las características generadas mediante TF-IDF.

El modelo aprendió patrones importantes dentro de los textos procesados y posteriormente realizó predicciones sobre los datos de prueba.

Finalmente, se mostraron las primeras predicciones obtenidas para validar el funcionamiento del modelo.

In [ ]:
lr_tfidf = LogisticRegression(max_iter=1000)

lr_tfidf.fit(X_train_tfidf, y_train)

pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

print(pred_lr_tfidf[:10])

[1 1 1 1 1 1 0 1 0 1]


## Evaluación del modelo de Regresión Logística con TF-IDF

Finalmente, se evaluó el desempeño del modelo de Regresión Logística utilizando la representación TF-IDF.

El modelo obtuvo una exactitud aproximada del 83%, mostrando un mejor rendimiento en comparación con otros modelos probados anteriormente.

También se analizaron métricas como precisión, recall y f1-score, además de la matriz de confusión, para comprender el comportamiento del modelo en la clasificación de sentimientos positivos y negativos.

In [ ]:
print("Accuracy:")
print(accuracy_score(y_test, pred_lr_tfidf))

print("\nClassification Report:")
print(classification_report(y_test, pred_lr_tfidf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred_lr_tfidf))

Accuracy:
0.8304904051172708

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.39      0.53       454
           1       0.83      0.97      0.90      1422

    accuracy                           0.83      1876
   macro avg       0.82      0.68      0.71      1876
weighted avg       0.83      0.83      0.81      1876


Confusion Matrix:
[[ 177  277]
 [  41 1381]]
